In [ ]:
"""
Script para filtrar setores censitários do município de São Paulo (SP capital)
a partir das tabelas de Agregados por Setores Censitários do IBGE (Censo).

Tabelas esperadas:
  - Domicilio01_SP.csv  (Características dos Domicílios 1)
  - Domicilio02_SP.csv  (Características dos Domicílios 2)
  - Domicilio03_SP.csv  (Características dos Domicílios 3)
  - Basico_SP.csv       (IBGE Básico)

Ajuste os nomes/caminhos dos arquivos na seção CONFIGURAÇÃO abaixo.
"""

import os
import pandas as pd

# ============================================================
# CONFIGURAÇÃO — ajuste aqui os caminhos dos seus arquivos
# ============================================================

# Pasta onde estão os CSVs.
# Caminho relativo: assume que existe uma pasta "data/" no mesmo nível deste notebook.
# Assim o código funciona em qualquer computador que clonar o repositório do GitHub,
# sem precisar editar caminhos pessoais.
# obs: em notebooks (.ipynb) usamos os.getcwd() em vez de __file__, pois __file__
# não é confiável dentro do Jupyter.
PASTA_DADOS = os.path.join(os.getcwd(), "data")

ARQUIVOS = {
    "domicilio1": os.path.join(PASTA_DADOS, "domicilio1.csv"),
    "domicilio2": os.path.join(PASTA_DADOS, "domicilio2.csv"),
    "domicilio3": os.path.join(PASTA_DADOS, "domicilio3.csv"),
    "basico":      os.path.join(PASTA_DADOS, "basico.csv"),
}

# Código do município de São Paulo (capital) — 7 primeiros dígitos do setor
CODIGO_MUNICIPIO_SP = "3550308"

# Parâmetros de leitura (os arquivos do IBGE costumam vir assim)
SEP = ";"
ENCODING = "latin1"

# Pasta de saída (salvando também dentro do Drive, na mesma pasta dos dados)
SAIDA = os.path.join(PASTA_DADOS, "setores_censitarios_sp_capital.csv")


# ============================================================
# FUNÇÕES
# ============================================================

def encontrar_coluna_setor(df: pd.DataFrame) -> str:
    """Encontra automaticamente a coluna que representa o código do setor censitário."""
    candidatos = [c for c in df.columns if "setor" in c.lower()]
    if not candidatos:
        raise ValueError(
            f"Não encontrei nenhuma coluna com 'setor' no nome. Colunas disponíveis: {list(df.columns)}"
        )
    # Prioriza colunas de código (evita pegar colunas de nome/descrição, se houver)
    for c in candidatos:
        if "cd" in c.lower() or "cod" in c.lower():
            return c
    return candidatos[0]


def carregar_e_filtrar(caminho: str, codigo_municipio: str) -> pd.DataFrame:
    """Lê um CSV do IBGE e filtra apenas os setores do município desejado."""
    df = pd.read_csv(caminho, sep=SEP, encoding=ENCODING, dtype=str)

    col_setor = encontrar_coluna_setor(df)

    # Garante que o código do setor está como string, sem espaços
    df[col_setor] = df[col_setor].astype(str).str.strip()

    # Os 7 primeiros dígitos do setor = código do município
    df["cd_municipio"] = df[col_setor].str[:7]

    filtrado = df[df["cd_municipio"] == codigo_municipio].copy()

    print(f"{caminho}: {len(df)} setores no total -> {len(filtrado)} setores em SP capital "
          f"(coluna de setor detectada: '{col_setor}')")

    # Renomeia a coluna de setor para um nome padrão, facilitando o merge depois
    filtrado = filtrado.rename(columns={col_setor: "cd_setor"})

    return filtrado


def main():
    if not os.path.isdir(PASTA_DADOS):
        raise FileNotFoundError(
            f"Pasta não encontrada: {PASTA_DADOS}\n"
            "Ajuste a variável PASTA_DADOS no topo do script para o caminho correto "
            "no seu computador (ex: onde o Google Drive está sincronizado localmente)."
        )

    tabelas_filtradas = {}

    for nome, caminho in ARQUIVOS.items():
        tabelas_filtradas[nome] = carregar_e_filtrar(caminho, CODIGO_MUNICIPIO_SP)

    # ------------------------------------------------------------
    # Merge de todas as tabelas pelo código do setor censitário
    # ------------------------------------------------------------
    nomes = list(tabelas_filtradas.keys())
    resultado = tabelas_filtradas[nomes[0]]

    for nome in nomes[1:]:
        df_outro = tabelas_filtradas[nome]

        # Remove colunas duplicadas (ex: cd_municipio, nome do município, etc.)
        colunas_repetidas = [c for c in df_outro.columns
                              if c in resultado.columns and c != "cd_setor"]
        df_outro = df_outro.drop(columns=colunas_repetidas)

        resultado = resultado.merge(df_outro, on="cd_setor", how="inner")

    print(f"\nTotal de setores após o merge das {len(nomes)} tabelas: {len(resultado)}")

    resultado.to_csv(SAIDA, index=False, sep=SEP, encoding=ENCODING)
    print(f"Arquivo final salvo em: {SAIDA}")


if __name__ == "__main__":
    main()

In [ ]:
with open("data/setores_censitarios_sp_capital.csv", "r", encoding="latin1") as f:
    print(f.readline())
    

In [ ]:
import pandas as pd
df_basico = pd.read_csv("data/basico.csv", sep=";", encoding="latin1", nrows=5)
print(df_basico.columns.tolist())

In [ ]:
# Um setor "pertence" a uma favela/comunidade quando CD_FCU não é nulo
df["em_favela"] = df["CD_FCU"].notna()

# População do setor (v0005 = moradores em domicílios permanentes) 
# atribuída como "população em favela" apenas nos setores marcados
df["N_POP_DPPO"] = df["v0005"].where(df["em_favela"], 0)

In [ ]:
# 1. DATA PROCESSING AND BASE CONSOLIDATION (RECORTE RMSP - CHAVE CD_SETOR)

import os
import pandas as pd
from google.colab import drive

# 1. Monta o Google Drive
drive.mount('/content/drive', force_remount=False)


# 2. Localização Dinâmica da Pasta
def encontrar_pasta_projeto():
    raiz = "/content/drive/MyDrive"
    for root, dirs, files in os.walk(raiz):
        if "AD_ARARAQUARA" in root:
            for d in dirs:
                if "data" in d.strip().lower():
                    return os.path.join(root, d)
            return root
    return None


pasta_real = encontrar_pasta_projeto()


def obter_caminho_arquivo(nome_arquivo):
    for arq in os.listdir(pasta_real):
        if arq.strip().lower() == nome_arquivo.strip().lower():
            return os.path.join(pasta_real, arq)
    raise FileNotFoundError(
        f"❌ Arquivo '{nome_arquivo}' não foi encontrado em '{pasta_real}'."
    )


print(f"✅ Pasta localizada em: '{pasta_real}'")
print("⏳ Carregando os arquivos CSV...")

# 3. Data Loading
dom1 = pd.read_csv(
    obter_caminho_arquivo("ACD_ar1.csv"), encoding="latin1", sep=";"
).drop(columns=["NM_MUN", "NM_UF", "CD_UF", "CD_MUN"], errors="ignore")

dom2 = pd.read_csv(
    obter_caminho_arquivo("ACD_ar2.csv"), encoding="latin1", sep=";"
).drop(columns=["NM_MUN", "NM_UF", "CD_UF", "CD_MUN"], errors="ignore")

dom3 = pd.read_csv(
    obter_caminho_arquivo("ACD_ar3.csv"), encoding="latin1", sep=";"
).drop(columns=["NM_MUN", "NM_UF", "CD_UF", "CD_MUN"], errors="ignore")

basico = pd.read_csv(
    obter_caminho_arquivo("ACD_basico_ar.csv"),
    encoding="latin1",
    sep=";",
)

# 4. Data Cleaning na tabela 'basico'
columns_to_remove = [
    "CD_REGIAO",
    "NM_REGIAO",
    "CD_UF",
    "NM_UF",
    "CD_NU",
    "NM_NU",
    "CD_AGLOM",
    "NM_AGLOM",
    "CD_RGINT",
    "NM_RGINT",
    "CD_RGI",
    "NM_RGI",
    "CD_CONCURB",
    "NM_CONCURB",
    "AREA_KM2",
    "v0001",
    "v0002",
    "v0003",
    "v0004",
    "v0006",
]
basico = basico.drop(columns=columns_to_remove, errors="ignore")

# 5. Data Standardization (Padronização estrita da chave CD_SETOR)
print("🔧 Padronizando a chave 'CD_SETOR' em todas as tabelas...")

for df_temp in [basico, dom1, dom2, dom3]:
    if "CD_SETOR" in df_temp.columns:
        df_temp["CD_SETOR"] = df_temp["CD_SETOR"].astype(str).str.strip()
    else:
        raise KeyError(
            "❌ A coluna 'CD_SETOR' não foi encontrada em uma das tabelas!"
        )

# 6. Data Merge pela chave CD_SETOR
print("🔄 Realizando o Merge das bases pela chave 'CD_SETOR'...")

df = (
    basico.merge(dom1, on="CD_SETOR", how="left")
    .merge(dom2, on="CD_SETOR", how="left")
    .merge(dom3, on="CD_SETOR", how="left")
)

print(
    f"\n🎉 RMSP dataset consolidado com sucesso!"
)
print(f"📊 Registros/Setores: {df.shape[0]} | Variáveis totais: {df.shape[1]}")

# 7. Salvando a base final
caminho_saida = os.path.join(pasta_real, "base_final_arara.csv")
df.to_csv(caminho_saida, index=False, sep=";", encoding="utf-8")
print(f"💾 Arquivo salvo com sucesso em:\n   {caminho_saida}")